# Guards vs No Guards Calibrated Explanations for Binary Classification
## Explanation Time Analysis

Author: Tuwe Löfström (tuwe.lofstrom@ju.se)  
Copyright 2025 Tuwe Löfström  
License: BSD 3 clause

### 1. Import packages

In [23]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
import pickle
import pandas as pd
import numpy as np
from scipy import stats as st

### 2 Import results from the pickled result file

In [25]:
with open('../results_guards_ablation.pkl', 'rb') as f:
    results = pickle.load(f)
data_characteristics = {'colic': 60, 
                        'creditA': 43, 
                        'diabetes': 9, 
                        'german': 28, 
                        'haberman': 4, 
                        'haberman': 4,
                        'heartC': 23,
                        'heartH': 21,
                        'heartS': 14,
                        'hepati': 20,
                        'iono': 34,
                        'je4042': 9,
                        'je4243': 9, 
                        'kc1': 22,
                        'kc2': 22,
                        'kc3': 40,
                        'liver': 7,
                        'pc1req': 9,
                        'pc4': 38,
                        'sonar': 61,
                        'spect': 23,
                        'spectf': 45,
                        'transfusion': 5,
                        'ttt': 28,
                        'vote': 17,
                        'wbc': 10,}

In [26]:
print(results.keys())

dict_keys(['alphas', 'n_clusters_options', 'covariances', 'use_martingales', 'severities', 'noise_type', 'test_size', 'pc1req', 'haberman', 'hepati', 'transfusion', 'spect', 'heartS', 'heartH', 'heartC', 'je4243', 'vote', 'kc2', 'wbc', 'kc3', 'creditA', 'diabetes', 'iono', 'liver', 'je4042', 'sonar', 'spectf', 'german', 'ttt', 'colic', 'pc4', 'kc1'])


In [27]:
print(timer.keys())

dict_keys(['ce_init', 'ce_explain', 'pce_guarded_init', 'pce_guarded_explain', 'pce_unguarded_init', 'pce_unguarded_explain'])


In [28]:
timer = results['pc1req']['RF']['timer']
pce_init = timer['pce_guarded_init']
pce_explain = timer['pce_guarded_explain']

# Remove parameter keys and other non-dataset keys from results and store in a new dict
filtered_results = {k: v for k, v in results.items() if k not in ['alphas', 'n_clusters_options', 'covariances', 'use_martingales', 'severities', 'noise_type', 'scale_factor', 'test_size']}

# Create a DataFrame for pce_init
df_pce = pd.DataFrame.from_dict({(i, j, k, l, m, n): filtered_results[i][j]['timer']['pce_guarded_init'][k][l][m][n]
                                   for i in filtered_results.keys() 
                                   for j in filtered_results[i].keys() 
                                   for k in filtered_results[i][j]['timer']['pce_guarded_init'].keys()
                                   for l in filtered_results[i][j]['timer']['pce_guarded_init'][k].keys()
                                   for m in filtered_results[i][j]['timer']['pce_guarded_init'][k][l].keys()
                                   for n in filtered_results[i][j]['timer']['pce_guarded_init'][k][l][m].keys()},
                                  orient='index')
df_pce.reset_index(inplace=True)
df_pce.columns = ['dataset - algorithm - alpha - n_clusters - covariance - use_martingale', 'init_time']

# Create a DataFrame for pce_explain
df_pce_explain = pd.DataFrame.from_dict({(i, j, k, l, m, n): filtered_results[i][j]['timer']['pce_guarded_explain'][k][l][m][n]
                                   for i in filtered_results.keys() 
                                   for j in filtered_results[i].keys() 
                                   for k in filtered_results[i][j]['timer']['pce_guarded_explain'].keys()
                                   for l in filtered_results[i][j]['timer']['pce_guarded_explain'][k].keys()
                                   for m in filtered_results[i][j]['timer']['pce_guarded_explain'][k][l].keys()
                                   for n in filtered_results[i][j]['timer']['pce_guarded_explain'][k][l][m].keys()},
                                  orient='index')
df_pce_explain.reset_index(inplace=True)
df_pce_explain.columns = ['dataset - algorithm - alpha - n_clusters - covariance - use_martingale', 'explain_time']

# Merge the two DataFrames on 'dataset - algorithm - alpha - n_clusters - covariance - use_martingale'
df_pce = pd.merge(df_pce, df_pce_explain, on='dataset - algorithm - alpha - n_clusters - covariance - use_martingale')
print(df_pce)

     dataset - algorithm - alpha - n_clusters - covariance - use_martingale  \
0                   (pc1req, xGB, 0.05, 3, diag, False)                       
1                    (pc1req, xGB, 0.05, 3, diag, True)                       
2                   (pc1req, xGB, 0.05, 3, full, False)                       
3                    (pc1req, xGB, 0.05, 3, full, True)                       
4                   (pc1req, xGB, 0.05, 5, diag, False)                       
...                                                 ...                       
1795                      (kc1, RF, 0.2, 5, full, True)                       
1796                    (kc1, RF, 0.2, 10, diag, False)                       
1797                     (kc1, RF, 0.2, 10, diag, True)                       
1798                    (kc1, RF, 0.2, 10, full, False)                       
1799                     (kc1, RF, 0.2, 10, full, True)                       

      init_time  explain_time  
0      0.008636    

In [29]:
criteria = {'alpha': 2, 'n_clusters': 3, 'covariance': 4, 'use_martingale': 5}
# BEGIN: Calculate average time per scale_factor
for c, idx in criteria.items():
    df_pce[c] = df_pce['dataset - algorithm - alpha - n_clusters - covariance - use_martingale'].apply(lambda x: x[idx])
    average_time_per_scale_factor = df_pce.groupby(c)[['init_time','explain_time']].mean()
    print(average_time_per_scale_factor)
# END: Calculate average time per scale_factor

       init_time  explain_time
alpha                         
0.05    0.000198      0.005966
0.10    0.000183      0.005998
0.20    0.000181      0.006093
            init_time  explain_time
n_clusters                         
3            0.000187      0.006035
5            0.000181      0.006085
10           0.000194      0.005936
            init_time  explain_time
covariance                         
diag         0.000194      0.006015
full         0.000181      0.006023
                init_time  explain_time
use_martingale                         
False            0.000191      0.006017
True             0.000184      0.006021
            init_time  explain_time
n_clusters                         
3            0.000187      0.006035
5            0.000181      0.006085
10           0.000194      0.005936
            init_time  explain_time
covariance                         
diag         0.000194      0.006015
full         0.000181      0.006023
                init_time  explain_ti

In [30]:
# Create a DataFrame for pce_init
df_ce = pd.DataFrame.from_dict({(i, j): filtered_results[i][j]['timer']['ce_init']
                                   for i in filtered_results.keys() 
                                   for j in filtered_results[i].keys()},
                                  orient='index')
df_ce.reset_index(inplace=True)
df_ce.columns = ['dataset - algorithm', 'init_time']

# Create a DataFrame for ce_init
df_ce_explain = pd.DataFrame.from_dict({(i, j): filtered_results[i][j]['timer']['ce_explain']
                                   for i in filtered_results.keys() 
                                   for j in filtered_results[i].keys()},
                                  orient='index')
df_ce_explain.reset_index(inplace=True)
df_ce_explain.columns = ['dataset - algorithm', 'explain_time']

# Merge the two DataFrames on 'dataset - algorithm'
df_ce = pd.merge(df_ce, df_ce_explain, on='dataset - algorithm')
# print(df_ce)


In [31]:
# Pivot the DataFrame to get datasets as rows and criteria and algorithm as columns
pivot_df = df_ce.pivot_table(index='dataset - algorithm', 
                              values=['init_time', 'explain_time'], 
                              aggfunc='mean')

# Reset the index to make it easier to work with
pivot_df.reset_index(inplace=True)

# Split the 'dataset - algorithm - scale_factor - severity - noise_type' column into separate columns
pivot_df[['dataset', 'algorithm']] = pd.DataFrame(pivot_df['dataset - algorithm'].tolist(), index=pivot_df.index)

# Drop the original combined column
pivot_df.drop(columns=['dataset - algorithm'], inplace=True)

# Pivot again to get the desired format
final_df = pivot_df.pivot_table(index='dataset', 
                                columns=['algorithm'], 
                                values=['init_time', 'explain_time'], 
                                aggfunc='mean')

final_df['Num Features'] = data_characteristics.values()
# Print the final DataFrame
display(final_df)

final_df.to_csv('ce_init_explain_time.csv')


explain_time           init_time           Num Features
algorithm             RF       xGB        RF       xGB             
dataset                                                            
colic           0.013719  0.009509  0.000224  0.000090           60
creditA         0.006967  0.004994  0.000077  0.000046           43
diabetes        0.002535  0.002935  0.000057  0.000036            9
german          0.003409  0.002652  0.000054  0.000036           28
haberman        0.001274  0.000930  0.000127  0.000084            4
heartC          0.005959  0.003870  0.000133  0.000055           23
heartH          0.004546  0.002854  0.000178  0.000137           21
heartS          0.003399  0.002144  0.000157  0.000089           14
hepati          0.006171  0.002783  0.000326  0.000141           20
iono            0.011511  0.008856  0.000145  0.000134           34
je4042          0.003169  0.002087  0.000160  0.000103            9
je4243          0.002834  0.002050  0.000176  0.000113            9
kc1             0.005952  0.005773  0.000041  0.000023           22
kc2             0.008709  0.005486  0.000190  0.000083           22
kc3             0.012711  0.009316  0.000153  0.000082           40
liver           0.002509  0.001682  0.000145  0.000088            7
pc1req          0.003044  0.001236  0.000346  0.000313            9
pc4             0.011150  0.016725  0.000047  0.000041           38
sonar           0.022961  0.019347  0.000209  0.000139           61
spect           0.003941  0.001832  0.000163  0.000164           23
spectf          0.024437  0.013754  0.000168  0.000112           45
transfusion     0.001471  0.001012  0.000080  0.000048            5
ttt             0.007264  0.002836  0.000157  0.000133           28
vote            0.002330  0.001329  0.000142  0.000071           17
wbc             0.002840  0.002195  0.000086  0.000056           10

In [32]:
mean_df = pivot_df.pivot_table(columns=['algorithm'], 
                                values=['init_time', 'explain_time'], 
                                aggfunc='mean')
display(mean_df)

algorithm,RF,xGB
explain_time,0.006993,0.005127
init_time,0.000150,0.000097


In [33]:
# Pivot the DataFrame to get datasets as rows and criteria and algorithm as columns
pivot_df = df_pce.pivot_table(index='dataset - algorithm - alpha - n_clusters - covariance - use_martingale', 
                              values=['init_time', 'explain_time'], 
                              aggfunc='mean')

# Reset the index to make it easier to work with
pivot_df.reset_index(inplace=True)

# Split the 'dataset - algorithm - alpha - n_clusters - covariance - use_martingale' column into separate columns
pivot_df[['dataset', 'algorithm', 'alpha', 'n_clusters', 'covariance', 'use_martingale']] = pd.DataFrame(pivot_df['dataset - algorithm - alpha - n_clusters - covariance - use_martingale'].tolist(), index=pivot_df.index)

# Drop the original combined column
pivot_df.drop(columns=['dataset - algorithm - alpha - n_clusters - covariance - use_martingale'], inplace=True)

filtered_df = pivot_df

# Pivot again to get the desired format
final_df = filtered_df.pivot_table(index='dataset', 
                                   columns=['algorithm', 'covariance', 'alpha', 'n_clusters', 'use_martingale'], 
                                   values=['explain_time', 'init_time', ], 
                                   aggfunc='mean')

# Print the final DataFrame
# display(final_df)
final_df.to_csv('explain_time.csv')

# Filter the DataFrame to include only rows where covariance is 'diag'
filtered_df = pivot_df[pivot_df['covariance'] == 'diag']

# Pivot again to get the desired format
final_df = filtered_df.pivot_table(index='dataset', 
                                   columns=['covariance', 'alpha', 'n_clusters', 'use_martingale'], 
                                   values=['explain_time'], 
                                   aggfunc='mean')

# Print the final DataFrame
display(final_df)

# Filter the DataFrame to include only rows where covariance is 'full'
filtered_df = pivot_df[pivot_df['covariance'] == 'full']

# Pivot again to get the desired format
final_df = filtered_df.pivot_table(index='dataset', 
                                   columns=['covariance', 'alpha', 'n_clusters', 'use_martingale'], 
                                   values=['explain_time'], 
                                   aggfunc='mean')

# Print the final DataFrame
display(final_df)




explain_time                                                    \
covariance             diag                                                     
alpha                  0.05                                                     
n_clusters               3                   5                   10             
use_martingale        False     True      False     True      False     True    
dataset                                                                         
colic              0.011529  0.012606  0.011028  0.010701  0.010819  0.011079   
creditA            0.005553  0.005890  0.005708  0.005786  0.005450  0.006068   
diabetes           0.002593  0.002716  0.002318  0.002820  0.002693  0.002383   
german             0.003392  0.003116  0.003545  0.003940  0.003073  0.003841   
haberman           0.001197  0.001427  0.001293  0.001208  0.001109  0.001140   
heartC             0.005150  0.004606  0.005727  0.005212  0.004567  0.004757   
heartH             0.003797  0.003808  0.003791  0.003829  0.003560  0.004134   
heartS             0.002872  0.002875  0.002924  0.002943  0.003205  0.003179   
hepati             0.004868  0.004792  0.003872  0.004784  0.004402  0.005130   
iono               0.009819  0.010333  0.009629  0.009716  0.010490  0.010000   
je4042             0.002641  0.003029  0.002823  0.002542  0.002971  0.002561   
je4243             0.002638  0.002545  0.002090  0.002331  0.002743  0.002751   
kc1                0.006317  0.006395  0.005893  0.006649  0.007232  0.006486   
kc2                0.006509  0.005793  0.006408  0.006938  0.007867  0.006305   
kc3                0.012059  0.010735  0.010707  0.011858  0.010970  0.010981   
liver              0.002033  0.002145  0.002201  0.002031  0.001995  0.001995   
pc1req             0.002115  0.001986  0.001972  0.001968  0.002145  0.002110   
pc4                0.014099  0.012515  0.011581  0.013040  0.011647  0.011563   
sonar              0.021213  0.019913  0.021090  0.022775  0.020701  0.020985   
spect              0.002867  0.002729  0.002837  0.002862  0.003070  0.003042   
spectf             0.016133  0.016407  0.019732  0.016346  0.015670  0.015687   
transfusion        0.001144  0.001389  0.001305  0.001271  0.001339  0.001363   
ttt                0.006170  0.006209  0.006827  0.008266  0.006339  0.007326   
vote               0.002048  0.001884  0.002191  0.001775  0.001931  0.001761   
wbc                0.002755  0.002595  0.002673  0.002569  0.002547  0.002437   

                                                                            \
covariance                                                                   
alpha               0.10                                                     
n_clusters            3                   5                   10             
use_martingale     False     True      False     True      False     True    
dataset                                                                      
colic           0.010260  0.011676  0.011463  0.011674  0.012055  0.011114   
creditA         0.005771  0.005455  0.005534  0.006001  0.005696  0.005209   
diabetes        0.002727  0.002313  0.002633  0.002455  0.002865  0.002430   
german          0.003400  0.003565  0.003184  0.003612  0.003315  0.003636   
haberman        0.001213  0.001251  0.001181  0.001150  0.001218  0.001126   
heartC          0.004647  0.004916  0.004783  0.005045  0.004911  0.004810   
heartH          0.003432  0.003430  0.004271  0.005322  0.005103  0.004576   
heartS          0.004020  0.002805  0.002700  0.002726  0.002793  0.002763   
hepati          0.004395  0.004406  0.004273  0.004429  0.004140  0.004000   
iono            0.009826  0.010150  0.009894  0.009364  0.009571  0.010273   
je4042          0.002966  0.002840  0.002652  0.002719  0.002679  0.003141   
je4243          0.002094  0.002179  0.002511  0.002350  0.002720  0.002290   
kc1             0.005844  0.007318  0.008012  0.007148  0.006505  0.006267   
kc2             0.007

explain_time                                                    \
covariance             full                                                     
alpha                  0.05                                                     
n_clusters               3                   5                   10             
use_martingale        False     True      False     True      False     True    
dataset                                                                         
colic              0.010641  0.011033  0.010611  0.010204  0.009857  0.010587   
creditA            0.005345  0.005499  0.005713  0.005733  0.005740  0.005397   
diabetes           0.002426  0.002396  0.002444  0.002426  0.002721  0.002399   
german             0.003460  0.002940  0.003477  0.003506  0.003722  0.003993   
haberman           0.001204  0.001107  0.001503  0.001104  0.001220  0.001199   
heartC             0.005883  0.004848  0.005229  0.004408  0.004730  0.005349   
heartH             0.003970  0.003488  0.004110  0.004508  0.003632  0.003841   
heartS             0.003320  0.002912  0.002921  0.003060  0.002977  0.002875   
hepati             0.004299  0.004389  0.004144  0.004188  0.004523  0.004714   
iono               0.010277  0.010203  0.010092  0.009895  0.009875  0.010153   
je4042             0.003420  0.003018  0.002679  0.002600  0.002768  0.002815   
je4243             0.002431  0.002540  0.002314  0.002720  0.002662  0.002438   
kc1                0.005995  0.005915  0.006081  0.006969  0.006025  0.006659   
kc2                0.006127  0.006624  0.007541  0.007167  0.006295  0.006956   
kc3                0.010388  0.011312  0.010974  0.010914  0.010364  0.011741   
liver              0.002164  0.002352  0.001977  0.002023  0.001990  0.001928   
pc1req             0.002145  0.001993  0.001985  0.002121  0.002183  0.002221   
pc4                0.011882  0.011386  0.011745  0.012090  0.011533  0.011592   
sonar              0.022197  0.023357  0.020178  0.022642  0.021497  0.019909   
spect              0.002804  0.002693  0.003392  0.003667  0.003116  0.003031   
spectf             0.017773  0.017010  0.015213  0.014894  0.015277  0.013662   
transfusion        0.001255  0.001211  0.001306  0.001584  0.001313  0.001182   
ttt                0.006437  0.006308  0.007631  0.006960  0.006313  0.006239   
vote               0.002184  0.001876  0.001853  0.001925  0.001994  0.002144   
wbc                0.002608  0.002658  0.002443  0.002435  0.002525  0.002925   

                                                                            \
covariance                                                                   
alpha               0.10                                                     
n_clusters            3                   5                   10             
use_martingale     False     True      False     True      False     True    
dataset                                                                      
colic           0.010383  0.011077  0.011975  0.012753  0.011583  0.010479   
creditA         0.006023  0.005411  0.005922  0.005126  0.006432  0.005688   
diabetes        0.002439  0.002917  0.002442  0.002438  0.002394  0.002904   
german          0.003557  0.003491  0.003628  0.003050  0.003606  0.003752   
haberman        0.001337  0.001180  0.001157  0.001659  0.001130  0.001141   
heartC          0.005207  0.005019  0.005448  0.004836  0.004875  0.004999   
heartH          0.004045  0.004580  0.004135  0.004490  0.004432  0.003858   
heartS          0.003053  0.002903  0.002928  0.002772  0.002690  0.002756   
hepati          0.004177  0.004650  0.004288  0.004360  0.004120  0.003942   
iono            0.010073  0.009582  0.010013  0.011171  0.009871  0.009643   
je4042          0.003187  0.002598  0.002615  0.002551  0.002962  0.003107   
je4243          0.002098  0.002093  0.002832  0.002571  0.002299  0.002356   
kc1             0.006102  0.006335  0.006651  0.006500  0.006335  0.006406   
kc2             0.006

In [34]:
filtered_df = pivot_df[pivot_df['algorithm'] == 'RF']
mean_df = filtered_df.pivot_table(index='alpha',
                                columns=['covariance', 'n_clusters', 'use_martingale'], 
                                values=['explain_time'], 
                                aggfunc='mean')
display(mean_df)
# Assuming a baseline time, e.g., the ce explain time
# For speedup, perhaps compare to ce time
# But for now, just display
# speedup_df = 0.220064 / mean_df  # assuming 0.220064 is ce time or something
# speedup_df.to_csv('speedup.csv')

mean_df_init = filtered_df.pivot_table(index='alpha',
                                columns=['covariance', 'n_clusters', 'use_martingale'], 
                                values=['init_time'], 
                                aggfunc='mean')
display(mean_df_init)
# speeddown_df = mean_df_init / 0.000080  # assuming 0.000080 is ce init time
# speeddown_df.to_csv('speeddown_init.csv')


explain_time                                                    \
covariance             diag                                                     
n_clusters               3                   5                   10             
use_martingale        False     True      False     True      False     True    
alpha                                                                           
0.05               0.006956  0.006768  0.007005  0.006973  0.006909  0.006823   
0.10               0.006795  0.007020  0.006662  0.006643  0.007093  0.006846   
0.20               0.007189  0.007351  0.007091  0.007301  0.006844  0.006637   

                                                                            
covariance          full                                                    
n_clusters            3                   5                   10            
use_martingale     False     True      False     True      False     True   
alpha                                                                       
0.05            0.007009  0.006917  0.006752  0.006850  0.006750  0.006864  
0.10            0.007075  0.006916  0.007081  0.006944  0.006850  0.006687  
0.20            0.007192  0.007483  0.007411  0.007272  0.006857  0.007101

init_time                                                    \
covariance          diag                                                     
n_clusters            3                   5                   10             
use_martingale     False     True      False     True      False     True    
alpha                                                                        
0.05            0.000202  0.000209  0.000218  0.000208  0.000216  0.000226   
0.10            0.000210  0.000204  0.000207  0.000204  0.000225  0.000227   
0.20            0.000201  0.000215  0.000211  0.000197  0.000228  0.000234   

                                                                            
covariance          full                                                    
n_clusters            3                   5                   10            
use_martingale     False     True      False     True      False     True   
alpha                                                                       
0.05            0.000208  0.000220  0.000221  0.000198  0.000221  0.000221  
0.10            0.000196  0.000196  0.000200  0.000211  0.000221  0.000217  
0.20            0.000181  0.000209  0.000210  0.000208  0.000225  0.000234